# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses :**
- **Test #1 :** $H_0 =$ la note client (`review_score`) a la même distribution pour les commandes livrées en retard et pour celles livrées à l'heure / $H_1 =$ les commandes en retard reçoivent des notes plus basses *(Justification du choix du test : deux groupes indépendants, variable ordinale (1 à 5) non gaussienne, donc **Mann-Whitney U unilatéral** plutôt qu'un test t)*
- **Test #2 :** $H_0 =$ aucune corrélation monotone entre le délai de livraison (`delivery_time_days`) et la note client / $H_1 =$ plus le délai est long, plus la note baisse *(Justification du choix du test : deux variables non gaussiennes (skewness du délai = 3,83), on teste un lien monotone, donc **corrélation de rangs de Spearman**)*

**Résultats et conclusions (95 824 commandes notées, dont 6,7 % livrées en retard) :**
- **Test #1 (Mann-Whitney U) :** note médiane de **1 en cas de retard contre 5 à l'heure** (moyennes 2,27 contre 4,29). La p-value est inférieure à 10⁻³⁰⁰ (affichée 0 par SciPy) : on **rejette H0**. La taille d'effet (rank-biserial = −0,64) est **forte**, donc la différence n'est pas seulement significative, elle est importante en pratique. 53,8 % des commandes en retard reçoivent la note 1, contre 6,6 % de celles livrées à l'heure.
- **Test #2 (Spearman) :** rho = **−0,235**, p-value < 10⁻³⁰⁰ : on **rejette H0**. Le lien est bien négatif (plus le délai est long, plus la note baisse), mais il est **faible à modéré**. Avec un échantillon de cette taille, presque tout lien est « significatif » : il faut regarder la force de la corrélation, pas seulement la p-value.
- **Lecture commune :** ce n'est pas la durée brute de livraison qui irrite le client, c'est le **non-respect de la date promise** (effet fort pour `is_late`, effet modeste pour le délai en jours). Pour la logistique, il vaut mieux tenir ses engagements que livrer vite. Limite : ces tests montrent une association, pas une causalité (un client mécontent du produit peut aussi noter bas).


In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"

# Projection côté Spark : seulement 4 colonnes, uniquement les commandes qui ont une note
df_sample = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .select("is_late", "delivery_time_days", "delivery_delay_days", "review_score")
    .filter(F.col("review_score").isNotNull())
    .toPandas()
)
assert df_sample is not None, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."
print(f"{len(df_sample):,} commandes notées | {df_sample['is_late'].mean():.1%} livrées en retard\n")

# --- Test #1 : Mann-Whitney U (notes des commandes en retard vs à l'heure) ---
late = df_sample.loc[df_sample["is_late"], "review_score"]
on_time = df_sample.loc[~df_sample["is_late"], "review_score"]
u_stat, p_val_1 = stats.mannwhitneyu(late, on_time, alternative="less")
rank_biserial = 2 * u_stat / (len(late) * len(on_time)) - 1  # taille d'effet : -1 (tout en faveur de on_time) à +1
print("Test #1 — Mann-Whitney U (H1 : notes en retard < notes à l'heure)")
print(f"   Note médiane   : en retard = {late.median():.1f} | à l'heure = {on_time.median():.1f}")
print(f"   Note moyenne   : en retard = {late.mean():.2f} | à l'heure = {on_time.mean():.2f}")
print(f"   U = {u_stat:,.0f} | p-value = {p_val_1:.3e} | taille d'effet (rank-biserial) = {rank_biserial:.3f}")
print(f"   Conclusion     : {'on rejette H0' if p_val_1 < 0.05 else 'on ne rejette pas H0'} (seuil 5 %)\n")

# --- Test #2 : corrélation de rangs de Spearman (délai de livraison vs note) ---
rho, p_val_2 = stats.spearmanr(df_sample["delivery_time_days"], df_sample["review_score"], alternative="less")
print("Test #2 — Spearman (H1 : plus le délai est long, plus la note baisse)")
print(f"   rho = {rho:.3f} | p-value = {p_val_2:.3e}")
print(f"   Conclusion     : {'on rejette H0' if p_val_2 < 0.05 else 'on ne rejette pas H0'} (seuil 5 %)")

# --- Lecture visuelle : part de notes 1-2 selon le retard ---
print("\nRépartition des notes (%) :")
print(pd.crosstab(df_sample["is_late"].map({True: "en retard", False: "à l'heure"}), df_sample["review_score"], normalize="index").mul(100).round(1))


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [ ]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/customer_segments"

# TODO 1 : Agréger les features comportementales par client (entité métier = customer_unique_id)
FEATURES = ["total_spend", "avg_order_value", "avg_delivery_days", "avg_review_score", "late_rate"]
entities_sdf = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .groupBy("customer_unique_id")
    .agg(
        F.count("*").alias("nb_orders"),
        F.round(F.sum("order_total_value"), 2).alias("total_spend"),
        F.round(F.avg("order_total_value"), 2).alias("avg_order_value"),
        F.round(F.avg("delivery_time_days"), 1).alias("avg_delivery_days"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(F.avg(F.col("is_late").cast("int")), 3).alias("late_rate"),
    )
    .filter(F.col("avg_review_score").isNotNull())
)
entities_pd = entities_sdf.toPandas()
assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."
print(f"{len(entities_pd):,} clients à segmenter")

# TODO 2 : Prétraitement (log1p sur les features positives asymétriques + StandardScaler) puis K-Means
X = entities_pd[FEATURES].copy()
for col in ["total_spend", "avg_order_value", "avg_delivery_days"]:
    X[col] = np.log1p(X[col])          # réduit l'asymétrie à droite avant standardisation
X_scaled = StandardScaler().fit_transform(X)   # chaque feature centrée-réduite -> poids égal dans la distance euclidienne

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Labellisation métier : on nomme chaque cluster à partir de son profil médian (règles, pas de numéro figé)
profile = entities_pd.groupby("cluster_id")[FEATURES].median()
spend_med = entities_pd["total_spend"].median()

def name_segment(row):
    if row["late_rate"] >= 0.5:
        return "Mécontents — livraison en retard"
    if row["avg_review_score"] <= 2.5:
        return "Déçus — livrés à l'heure mais mal notés"
    if row["total_spend"] >= spend_med:
        return "Satisfaits — gros panier"
    return "Satisfaits — petit panier"

segment_map = {cid: name_segment(profile.loc[cid]) for cid in profile.index}
entities_pd["segment"] = entities_pd["cluster_id"].map(segment_map)
assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

print("\nProfil médian par segment :")
print(entities_pd.groupby("segment")[FEATURES].median().join(
    entities_pd.groupby("segment").size().rename("nb_clients")))

# TODO 3 : Sauvegarder la table de segments en Delta Lake sur HDFS
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"\nTable de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [ ]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")